In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta

project_root = Path.cwd().parents[1]
sys.path.insert(0, str(project_root / 'common' / 'scripts'))
from metabase_client import MetabaseClient

# ── Parameters ──
NAV_DATE = '2026-03-04'
DEPOSITORY_FEE_RATE = 0.00025  # 0.025% p.a.

FUNDS = [
    {'code': 'TUV100', 'name': 'Tuleva III Samba Pensionifond', 'isin': 'EE3600001707'},
    {'code': 'TKF100', 'name': 'Tuleva Täiendav Kogumisfond', 'isin': 'EE0000003283'},
]

# One-off adjustments (outstanding invoices etc.)
ADJUSTMENTS = {
    'TUV100': -9676.30,  # unpaid February depositary fee invoice
}

client = MetabaseClient()
print(f'NAV date: {NAV_DATE}')
for f in FUNDS:
    adj = ADJUSTMENTS.get(f['code'], 0)
    adj_str = f'  (adj: {adj:+,.2f} EUR)' if adj else ''
    print(f'  {f["code"]}: {f["name"]}{adj_str}')

NAV date: 2026-03-04
  TUV100: Tuleva III Samba Pensionifond  (adj: -9,676.30 EUR)
  TKF100: Tuleva Täiendav Kogumisfond


In [2]:
# ── Fetch all data from Metabase ──
raw_positions = pd.DataFrame(client.execute_card(2296))
raw_index = pd.DataFrame(client.execute_card(2297))
raw_units = pd.DataFrame(client.execute_card(2298))
raw_registry = pd.DataFrame(client.execute_card(2299))

print(f'Card 2296 (positions):  {len(raw_positions)} rows')
print(f'Card 2297 (index):      {len(raw_index)} rows')
print(f'Card 2298 (units):      {len(raw_units)} rows')
print(f'Card 2299 (registry):   {len(raw_registry)} rows')

# ── Prepare per-fund data ──
fund_data = {}
for fund in FUNDS:
    fd = {}
    code = fund['code']

    all_positions = raw_positions[
        (raw_positions['Fund Code'] == code) &
        (raw_positions['Nav Date'] == NAV_DATE)
    ].copy()

    fd['all_positions'] = all_positions
    fd['securities'] = all_positions[all_positions['Account Type'] == 'SECURITY'].copy()
    fd['cash'] = all_positions[all_positions['Account Type'] == 'CASH'].copy()
    fd['receivables'] = all_positions[all_positions['Account Type'] == 'RECEIVABLES'].copy()
    fd['liabilities'] = all_positions[all_positions['Account Type'] == 'LIABILITY'].copy()
    fd['units_row'] = all_positions[all_positions['Account Type'] == 'UNITS'].copy()

    print(f'\n{code} on {NAV_DATE}: {len(all_positions)} rows  '
          f'({len(fd["securities"])} sec, {len(fd["cash"])} cash, '
          f'{len(fd["receivables"])} recv, {len(fd["liabilities"])} liab, {len(fd["units_row"])} units)')

    # Index data for this fund's ISIN — preferred source for reported NAV (correct dates)
    isin_key = fund['isin']
    fund_idx = raw_index[raw_index['Key'] == isin_key].copy()
    fund_idx['Date'] = pd.to_datetime(fund_idx['Date'])
    fund_idx = fund_idx.sort_values('Date', ascending=False)
    fd['index_nav_series'] = fund_idx

    # Determine reported NAV — prefer index data (exact date match)
    nav_dt_ts = pd.to_datetime(NAV_DATE)
    idx_on_date = fund_idx[fund_idx['Date'] == nav_dt_ts]
    if len(idx_on_date) > 0:
        fd['reported_nav'] = idx_on_date.iloc[0]['Value']
        fd['nav_source'] = 'index data'
    else:
        fd['reported_nav'] = None
        fd['nav_source'] = 'unavailable'

    # Previous reported NAV from index data
    older = fund_idx[fund_idx['Date'] < nav_dt_ts]
    prev_date = None
    if len(older) > 0:
        prev_date = older.iloc[0]['Date'].strftime('%Y-%m-%d')
        fd['prev_reported_nav'] = older.iloc[0]['Value']

    fd['prev_date'] = prev_date
    print(f'  Reported NAV source: {fd["nav_source"]}')
    if fd['reported_nav']:
        print(f'  Reported NAV: {fd["reported_nav"]:.6f}')
    if prev_date:
        print(f'  Prev reported NAV ({prev_date}): {fd["prev_reported_nav"]:.6f}')

    fund_reg = raw_registry[raw_registry['Isin'] == fund['isin']]
    if len(fund_reg) > 0:
        fd['mgmt_fee_rate'] = fund_reg.iloc[0]['Management Fee Rate']
        print(f'  Mgmt fee: {fd["mgmt_fee_rate"]*100:.3f}% p.a.')
    else:
        fd['mgmt_fee_rate'] = None
        print(f'  WARNING: Fund not found in registry!')

    print(f'  Depository fee: {DEPOSITORY_FEE_RATE*100:.3f}% p.a.')

    fund_data[code] = fd

Card 2296 (positions):  245 rows
Card 2297 (index):      531 rows
Card 2298 (units):      18 rows
Card 2299 (registry):   58 rows

TUV100 on 2026-03-04: 12 rows  (6 sec, 1 cash, 2 recv, 2 liab, 1 units)
  Reported NAV source: index data
  Reported NAV: 1.271000
  Prev reported NAV (2026-03-03): 1.266000
  Mgmt fee: 0.179% p.a.
  Depository fee: 0.025% p.a.

TKF100 on 2026-03-04: 15 rows  (9 sec, 1 cash, 2 recv, 2 liab, 1 units)
  Reported NAV source: index data
  Reported NAV: 0.998900
  Prev reported NAV (2026-03-03): 0.988300
  Mgmt fee: 0.150% p.a.
  Depository fee: 0.025% p.a.


## Fund Position

All holdings of the fund on the NAV date, split into securities (ETFs) and cash/accrual lines.

In [3]:
for fund in FUNDS:
    code = fund['code']
    fd = fund_data[code]
    securities = fd['securities']
    cash = fd['cash']
    receivables = fd['receivables']
    liabilities = fd['liabilities']
    units_row = fd['units_row']

    sec_display = securities[['Account ID', 'Account Name', 'Quantity', 'Market Price', 'Market Value']].copy()
    sec_display = sec_display.sort_values('Market Value', ascending=False)
    sec_total = sec_display['Market Value'].sum()

    cash_total = cash['Market Value'].sum()
    recv_total = receivables['Market Value'].sum()
    liab_total = liabilities['Market Value'].sum()
    other_rows = [
        {'Account ID': '', 'Account Name': 'Cash', 'Quantity': None, 'Market Price': None, 'Market Value': cash_total},
        {'Account ID': '', 'Account Name': 'Receivables', 'Quantity': None, 'Market Price': None, 'Market Value': recv_total},
        {'Account ID': '', 'Account Name': 'Liabilities', 'Quantity': None, 'Market Price': None, 'Market Value': liab_total},
    ]

    all_display = pd.concat([sec_display, pd.DataFrame(other_rows)], ignore_index=True)

    # Previous working day comparison
    available_dates = sorted(raw_positions[raw_positions['Fund Code'] == code]['Nav Date'].unique())
    nav_idx = available_dates.index(NAV_DATE) if NAV_DATE in available_dates else -1
    prev_nav_date = available_dates[nav_idx - 1] if nav_idx > 0 else None

    if prev_nav_date:
        prev_pos = raw_positions[
            (raw_positions['Fund Code'] == code) &
            (raw_positions['Nav Date'] == prev_nav_date)
        ]
        prev_sec = prev_pos[prev_pos['Account Type'] == 'SECURITY'].set_index('Account ID')['Market Value']
        prev_cash = prev_pos[prev_pos['Account Type'] == 'CASH']['Market Value'].sum()
        prev_recv = prev_pos[prev_pos['Account Type'] == 'RECEIVABLES']['Market Value'].sum()
        prev_liab = prev_pos[prev_pos['Account Type'] == 'LIABILITY']['Market Value'].sum()
        prev_by_type = {'Cash': prev_cash, 'Receivables': prev_recv, 'Liabilities': prev_liab}

        prev_values = []
        for _, row in all_display.iterrows():
            if row['Account ID'] and row['Account ID'] in prev_sec.index:
                prev_values.append(prev_sec[row['Account ID']])
            elif row['Account Name'] in prev_by_type:
                prev_values.append(prev_by_type[row['Account Name']])
            else:
                prev_values.append(None)

        all_display['Prev Value'] = prev_values
        all_display['Change %'] = (all_display['Market Value'] - all_display['Prev Value']) / all_display['Prev Value'].abs() * 100
    else:
        prev_pos = pd.DataFrame()
        all_display['Prev Value'] = None
        all_display['Change %'] = None

    gross_portfolio = all_display['Market Value'].abs().sum()
    all_display['% of Portfolio'] = all_display['Market Value'] / gross_portfolio * 100

    nav_components = sec_total + cash_total + recv_total + liab_total
    prev_nav_total = all_display['Prev Value'].sum() if prev_nav_date else None
    nav_change_pct = (nav_components - prev_nav_total) / abs(prev_nav_total) * 100 if prev_nav_total else None

    total_row = pd.DataFrame([{
        'Account ID': '', 'Account Name': 'Total net assets', 'Quantity': None,
        'Market Price': None, 'Market Value': nav_components,
        'Prev Value': prev_nav_total, 'Change %': nav_change_pct,
        '% of Portfolio': 100.0
    }])
    all_display = pd.concat([all_display, total_row], ignore_index=True)

    position_units = units_row.iloc[0]['Quantity'] if len(units_row) > 0 else None

    def bold_total(row):
        if row['Account Name'] == 'Total net assets':
            return ['font-weight: bold'] * len(row)
        return [''] * len(row)

    display(all_display[['Account ID', 'Account Name', 'Quantity', 'Market Price', 'Market Value', '% of Portfolio', 'Change %']].style
        .format({'Quantity': '{:,.2f}', 'Market Price': '{:.4f}', 'Market Value': '{:,.2f}',
                 '% of Portfolio': '{:.2f}%', 'Change %': '{:+.2f}%'}, na_rep='')
        .set_caption(f'{code} position as of {NAV_DATE} (vs {prev_nav_date})')
        .set_table_styles([{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '1.1em'), ('text-align', 'left'), ('padding-bottom', '8px')]}])
        .apply(bold_total, axis=1)
        .hide(axis='index'))

    # Store for later cells
    fd['sec_total'] = sec_total
    fd['cash_total'] = cash_total
    fd['recv_total'] = recv_total
    fd['liab_total'] = liab_total
    fd['nav_components'] = nav_components
    fd['prev_nav_date'] = prev_nav_date
    fd['prev_nav_total'] = prev_nav_total
    fd['prev_pos'] = prev_pos if prev_nav_date else pd.DataFrame()
    fd['position_units'] = position_units

Account ID,Account Name,Quantity,Market Price,Market Value,% of Portfolio,Change %
IE0009FT4LX4,CCF Developed World (ESG Screened) Index Fund Class X0,"9,039,054.83",15.4150,"139,337,030.19",28.68%,-0.73%
IE00BFG1TM61,iShares Developed World Screened Index Fund,"4,053,375.60",34.2700,"138,909,181.81",28.59%,-0.73%
IE00BFNM3G45,iShares MSCI USA Screened UCITS ETF,"9,135,951.00",12.1320,"110,837,357.53",22.81%,+2.20%
IE00BKPTWY98,iShares Emerging Market Screened Equity Index Fund Class Flexible,"3,848,906.59",13.3080,"51,221,248.90",10.54%,-2.91%
IE00BFNM3D14,iShares MSCI Europe Screened UCITS ETF,"3,585,405.00",10.2380,"36,707,376.39",7.56%,+1.47%
IE00BFNM3L97,iShares MSCI Japan Screened UCITS ETF,"520,422.00",7.6890,"4,001,524.76",0.82%,+3.19%
,Cash,,,"2,939,969.97",0.61%,+34.25%
,Receivables,,,"242,059.62",0.05%,-69.58%
,Liabilities,,,"-1,661,662.29",-0.34%,-307.29%
,Total net assets,,,"482,534,086.88",100.00%,-0.33%


Account ID,Account Name,Quantity,Market Price,Market Value,% of Portfolio,Change %
IE000F60HVH9,ICAV Amundi MSCI USA Screened UCITS ETF,"291,716.00",4.3880,"1,280,049.81",16.83%,+13.10%
IE00BJZ2DC62,Xtrackers MSCI USA Screened UCITS ETF,"23,687.00",50.7600,"1,202,352.12",15.81%,+1.16%
IE00BFG1TM61,iShares Developed World Screened Index Fund,"34,096.56",34.2700,"1,168,489.11",15.37%,-0.73%
IE000O58J820,Vanguard ESG North America All Cap UCITS ETF,"153,949.00",6.9320,"1,067,174.47",14.04%,+1.14%
LU1291099718,BNP Paribas Easy MSCI EUROPE MIN TE UCITS ETF,"45,082.00",19.1340,"862,598.99",11.34%,+1.35%
IE00BMDBMY19,Invesco MSCI Emerging Markets Universal Screened UCITS ETF Acc,"15,543.00",42.7050,"663,763.82",8.73%,+2.09%
LU1291102447,BNP Paribas Easy MSCI Japan Min TE UCITS ETF,"11,704.00",18.6040,"217,741.22",2.86%,+3.02%
LU1291106356,BNP Paribas Easy MSCI Pacific ex Japan Min TE UCITS ETF,"6,719.00",16.5260,"111,038.19",1.46%,+0.43%
LU0476289540,Xtrackers MSCI Canada Screened UCITS ETF,908.00,106.2800,"96,502.24",1.27%,+1.08%
,Cash,,,"563,890.95",7.42%,+14.11%


## Price Verification

Cross-checking fund position prices against independent index/provider data. These funds use NAV-date prices (no lag).

In [4]:
# ── Key mapping: ISIN → index keys per source ──
KEY_MAP = {
    # Shared: pooled BlackRock/iShares funds (TUV100 + TKF100)
    'IE0009FT4LX4': {
        'BlackRock': 'IE0009FT4LX4.BLACKROCK', 'EODHD': 'IE0009FT4LX4.EUFUND',
        'Morningstar': 'IE0009FT4LX4.MORNINGSTAR',
    },
    'IE00BFG1TM61': {
        'BlackRock': 'IE00BFG1TM61.BLACKROCK', 'EODHD': 'IE00BFG1TM61.EUFUND',
        'Morningstar': 'IE00BFG1TM61.MORNINGSTAR',
    },
    'IE00BKPTWY98': {
        'BlackRock': 'IE00BKPTWY98.BLACKROCK', 'EODHD': 'IE00BKPTWY98.EUFUND',
        'Morningstar': 'IE00BKPTWY98.MORNINGSTAR',
    },
    # TUV100 exchange-traded ETFs
    'IE00BFNM3G45': {
        'Exchange': 'IE00BFNM3G45.XETR', 'EODHD': 'SGAS.XETRA', 'Yahoo': 'SGAS.DE',
    },
    'IE00BFNM3D14': {
        'Exchange': 'IE00BFNM3D14.XETR', 'EODHD': 'SLMC.XETRA', 'Yahoo': 'SLMC.DE',
    },
    'IE00BFNM3L97': {
        'Exchange': 'IE00BFNM3L97.XETR', 'EODHD': 'SGAJ.XETRA', 'Yahoo': 'SGAJ.DE',
    },
    # TKF100 exchange-traded ETFs
    'IE00BJZ2DC62': {
        'Exchange': 'IE00BJZ2DC62.XETR', 'EODHD': 'XRSM.XETRA', 'Yahoo': 'XRSM.DE',
    },
    'IE000O58J820': {
        'Exchange': 'IE000O58J820.XETR', 'EODHD': 'V3YA.XETRA', 'Yahoo': 'V3YA.DE',
    },
    'IE000F60HVH9': {
        'Exchange': 'IE000F60HVH9.XPAR', 'EODHD': 'USAS.PA.EODHD', 'Yahoo': 'USAS.PA',
    },
    'LU1291106356': {
        'Exchange': 'LU1291106356.XETR', 'EODHD': 'PAC.XETRA', 'Yahoo': 'PAC.DE',
    },
    'IE00BMDBMY19': {
        'Exchange': 'IE00BMDBMY19.XETR', 'EODHD': 'ESGM.XETRA', 'Yahoo': 'ESGM.DE',
    },
    'LU1291102447': {
        'Exchange': 'LU1291102447.XETR', 'EODHD': 'EJAP.XETRA', 'Yahoo': 'EJAP.DE',
    },
    'LU1291099718': {
        'Exchange': 'LU1291099718.XETR', 'EODHD': 'EEUX.XETRA', 'Yahoo': 'EEUX.DE',
    },
    'LU0476289540': {
        'Exchange': 'LU0476289540.XETR', 'EODHD': 'D5BH.XETRA', 'Yahoo': 'D5BH.DE',
    },
}

# Positions where fund price is known stale — always reprice from providers,
# even from a single source (no consensus required)
ALWAYS_REPRICE = {'IE00BFG1TM61', 'IE0009FT4LX4', 'IE00BKPTWY98'}

SOURCES = ['BlackRock', 'Exchange', 'EODHD', 'Morningstar', 'Yahoo']

# Parse index data
idx = raw_index.copy()
idx['Date'] = pd.to_datetime(idx['Date'])
nav_dt = pd.to_datetime(NAV_DATE)

def latest_price_on_date(key, target_date):
    if key is None:
        return None
    rows = idx[(idx['Key'] == key) & (idx['Date'] == target_date)]
    if len(rows) == 0:
        return None
    return rows.iloc[0]['Value']

def find_consensus_price(alt_prices, threshold_pct=0.5):
    """Find consensus among ≥2 alternative prices that agree within threshold."""
    prices = [p for p in alt_prices if pd.notna(p)]
    if len(prices) < 2:
        return None
    for i in range(len(prices)):
        agreeing = [prices[i]]
        for j in range(len(prices)):
            if i != j and abs(prices[j] - prices[i]) / prices[i] * 100 <= threshold_pct:
                agreeing.append(prices[j])
        if len(agreeing) >= 2:
            return np.mean(agreeing)
    return None

for fund in FUNDS:
    code = fund['code']
    fd = fund_data[code]
    securities = fd['securities']

    verify_rows = []
    for _, sec in securities.sort_values('Market Value', ascending=False).iterrows():
        isin = sec['Account ID']
        mapping = KEY_MAP.get(isin, {})
        fund_price = sec['Market Price']

        # No lag — prices should be NAV date
        name = sec['Account Name']
        if isin in ALWAYS_REPRICE:
            name += ' *'
        row = {
            'ISIN': isin, 'Name': name, 'Fund': fund_price,
        }
        for src in SOURCES:
            row[src] = latest_price_on_date(mapping.get(src), nav_dt)
        verify_rows.append(row)

    verify_df = pd.DataFrame(verify_rows)

    def highlight_diff(row):
        fund_val = row['Fund']
        styles = [''] * len(row)
        for i, col in enumerate(row.index):
            if col in SOURCES and pd.notna(row[col]):
                diff_pct = abs(row[col] - fund_val) / fund_val * 100
                if diff_pct > 0.5:
                    styles[i] = 'background-color: #f8d7da'
                elif diff_pct > 0.01:
                    styles[i] = 'background-color: #fff3cd'
                else:
                    styles[i] = 'background-color: #d4edda'
        return styles

    if len(verify_df) > 0:
        display(verify_df.style
            .format({col: '{:.4f}' for col in ['Fund'] + SOURCES}, na_rep='—')
            .set_caption(f'{code} price verification (NAV date: {NAV_DATE})')
            .set_table_styles([{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '1.1em'), ('text-align', 'left'), ('padding-bottom', '8px')]}])
            .apply(highlight_diff, axis=1)
            .hide(axis='index'))

    n_ok = n_flag = n_nodata = 0
    for _, row in verify_df.iterrows():
        for src in SOURCES:
            val = row[src]
            if pd.isna(val) or val is None:
                n_nodata += 1
            elif abs(val - row['Fund']) / row['Fund'] * 100 > 0.5:
                n_flag += 1
            else:
                n_ok += 1

    # Compute repricing adjustment
    reprice_adj = 0.0
    for _, vrow in verify_df.iterrows():
        isin = vrow['ISIN']
        fund_price = vrow['Fund']
        alt_prices = [vrow[src] for src in SOURCES if src in vrow and pd.notna(vrow[src])]

        if isin in ALWAYS_REPRICE:
            # Fund price is known stale — use any available provider price
            nav_date_prices = [p for p in alt_prices if pd.notna(p)]
            if nav_date_prices:
                best_price = np.mean(nav_date_prices)
                qty_row = securities[securities['Account ID'] == isin]
                if len(qty_row) > 0:
                    reprice_adj += (best_price - fund_price) * qty_row.iloc[0]['Quantity']
        else:
            # Standard consensus repricing (≥2 sources agree, >0.5% diff)
            consensus = find_consensus_price(alt_prices)
            if consensus is not None and abs(consensus - fund_price) / fund_price * 100 > 0.5:
                qty_row = securities[securities['Account ID'] == isin]
                if len(qty_row) > 0:
                    reprice_adj += (consensus - fund_price) * qty_row.iloc[0]['Quantity']

    fd['verify_df'] = verify_df
    fd['n_ok'] = n_ok
    fd['n_flag'] = n_flag
    fd['n_nodata'] = n_nodata
    fd['reprice_adj'] = reprice_adj
    summary = f'{code}: {n_ok} OK  |  {n_flag} flagged  |  {n_nodata} no data'
    if abs(reprice_adj) > 0.01:
        summary += f'  |  reprice adj: {reprice_adj:+,.2f} EUR'
    print(summary)
    if any(vrow['ISIN'] in ALWAYS_REPRICE for _, vrow in verify_df.iterrows()):
        print(f'  (* = fund price is stale, always repriced from providers)')

ISIN,Name,Fund,BlackRock,Exchange,EODHD,Morningstar,Yahoo
IE0009FT4LX4,CCF Developed World (ESG Screened) Index Fund Class X0 *,15.4150,15.4300,—,—,15.4300,—
IE00BFG1TM61,iShares Developed World Screened Index Fund *,34.2700,—,—,—,34.3230,—
IE00BFNM3G45,iShares MSCI USA Screened UCITS ETF,12.1320,—,12.1320,12.1320,—,12.1320
IE00BKPTWY98,iShares Emerging Market Screened Equity Index Fund Class Flexible *,13.3080,—,—,—,12.7460,—
IE00BFNM3D14,iShares MSCI Europe Screened UCITS ETF,10.2380,—,10.2380,10.2380,—,10.2380
IE00BFNM3L97,iShares MSCI Japan Screened UCITS ETF,7.6890,—,7.6890,7.6890,—,7.6890


TUV100: 12 OK  |  1 flagged  |  17 no data  |  reprice adj: -1,812,670.77 EUR
  (* = fund price is stale, always repriced from providers)


ISIN,Name,Fund,BlackRock,Exchange,EODHD,Morningstar,Yahoo
IE000F60HVH9,ICAV Amundi MSCI USA Screened UCITS ETF,4.3880,—,4.3880,4.3880,—,4.3880
IE00BJZ2DC62,Xtrackers MSCI USA Screened UCITS ETF,50.7600,—,50.7600,50.7600,—,50.7600
IE00BFG1TM61,iShares Developed World Screened Index Fund *,34.2700,—,—,—,34.3230,—
IE000O58J820,Vanguard ESG North America All Cap UCITS ETF,6.9320,—,6.9320,6.9320,—,6.9320
LU1291099718,BNP Paribas Easy MSCI EUROPE MIN TE UCITS ETF,19.1340,—,19.1340,19.1340,—,19.1340
IE00BMDBMY19,Invesco MSCI Emerging Markets Universal Screened UCITS ETF Acc,42.7050,—,42.7050,42.7050,—,42.7050
LU1291102447,BNP Paribas Easy MSCI Japan Min TE UCITS ETF,18.6040,—,18.6040,18.6040,—,18.6040
LU1291106356,BNP Paribas Easy MSCI Pacific ex Japan Min TE UCITS ETF,16.5260,—,16.5260,16.5260,—,16.5260
LU0476289540,Xtrackers MSCI Canada Screened UCITS ETF,106.2800,—,106.2800,106.2800,—,106.2800


TKF100: 25 OK  |  0 flagged  |  20 no data  |  reprice adj: +1,807.12 EUR
  (* = fund price is stale, always repriced from providers)


## Net Asset Value Calculation

Computing NAV per unit from position data and comparing to the reported value. Fee accrual includes both management fee and depository bank fee (0.025% p.a.), accruing from the start of each month.

In [5]:
for fund in FUNDS:
    code = fund['code']
    fd = fund_data[code]
    nav_components = fd['nav_components']
    reprice_adj = fd.get('reprice_adj', 0)
    position_units = fd['position_units']
    mgmt_fee_rate = fd['mgmt_fee_rate']
    prev_nav_date = fd['prev_nav_date']
    prev_nav_total = fd['prev_nav_total']
    prev_pos = fd['prev_pos']
    adjustment = ADJUSTMENTS.get(code, 0)
    reported_nav = fd.get('reported_nav')
    nav_source = fd.get('nav_source', 'unavailable')

    # Always use repriced assets
    total_assets = nav_components + reprice_adj

    nav_dt = datetime.strptime(NAV_DATE, '%Y-%m-%d')
    day_of_month = nav_dt.day
    total_fee_rate = mgmt_fee_rate + DEPOSITORY_FEE_RATE
    mgmt_accrual = total_assets * mgmt_fee_rate / 365 * day_of_month
    depo_accrual = total_assets * DEPOSITORY_FEE_RATE / 365 * day_of_month
    total_accrual = mgmt_accrual + depo_accrual
    nav_after_fees = total_assets - total_accrual + adjustment
    computed_nav = nav_after_fees / position_units

    nav_diff_eur = (computed_nav - reported_nav) if reported_nav else None
    nav_diff_pct = (nav_diff_eur / reported_nav * 100) if reported_nav else None

    # Previous day
    prev_position_units = None
    prev_total_accrual = None
    prev_nav_after_fees = None
    prev_computed_nav = None
    if prev_nav_date and prev_nav_total:
        prev_units_rows = prev_pos[prev_pos['Account Type'] == 'UNITS'] if len(prev_pos) > 0 else pd.DataFrame()
        prev_position_units = prev_units_rows.iloc[0]['Quantity'] if len(prev_units_rows) > 0 else None
        prev_dt = datetime.strptime(prev_nav_date, '%Y-%m-%d')
        prev_total_accrual = prev_nav_total * total_fee_rate / 365 * prev_dt.day
        prev_nav_after_fees = prev_nav_total - prev_total_accrual
        if prev_position_units:
            prev_computed_nav = prev_nav_after_fees / prev_position_units

    def fmt_date(d):
        return datetime.strptime(d, '%Y-%m-%d').strftime('%d.%m.%Y') if d else 'Previous'

    def pct_change(cur, prev):
        if cur is not None and prev is not None and prev != 0:
            return f'{(cur - prev) / abs(prev) * 100:+.2f}%'
        return ''

    col_today = fmt_date(NAV_DATE)
    col_prev = fmt_date(prev_nav_date)

    table_rows = [
        {'': 'Total net assets as reported (EUR)', col_today: f'{nav_components:,.2f}', col_prev: f'{prev_nav_total:,.2f}' if prev_nav_total else '', 'Change': pct_change(nav_components, prev_nav_total)},
    ]
    if abs(reprice_adj) > 0.01:
        table_rows.append(
            {'': 'Repricing adjustment (EUR)', col_today: f'{reprice_adj:+,.2f}', col_prev: '', 'Change': ''},
        )
    table_rows += [
        {'': 'Total net assets repriced (EUR)', col_today: f'{total_assets:,.2f}', col_prev: '', 'Change': ''},
        {'': f'Accrued mgmt fee est. ({mgmt_fee_rate*100:.3f}% p.a.)', col_today: f'{-mgmt_accrual:+,.2f}', col_prev: '', 'Change': ''},
        {'': f'Accrued depository fee est. ({DEPOSITORY_FEE_RATE*100:.3f}% p.a.)', col_today: f'{-depo_accrual:+,.2f}', col_prev: '', 'Change': ''},
    ]
    if adjustment:
        table_rows.append(
            {'': 'Adjustment (outstanding invoices)', col_today: f'{adjustment:+,.2f}', col_prev: '', 'Change': ''},
        )
    table_rows += [
        {'': 'Net assets after fees (EUR)', col_today: f'{nav_after_fees:,.2f}', col_prev: f'{prev_nav_after_fees:,.2f}' if prev_nav_after_fees else '', 'Change': pct_change(nav_after_fees, prev_nav_after_fees)},
        {'': 'Units outstanding', col_today: f'{position_units:,.2f}', col_prev: f'{prev_position_units:,.2f}' if prev_position_units else '', 'Change': pct_change(position_units, prev_position_units)},
        {'': 'Computed NAV (EUR)', col_today: f'{computed_nav:.5f}', col_prev: f'{prev_computed_nav:.5f}' if prev_computed_nav else '', 'Change': pct_change(computed_nav, prev_computed_nav)},
    ]
    if reported_nav:
        nav_source_label = f' (from {nav_source})' if nav_source != 'units card' else ''
        table_rows += [
            {'': f'Reported NAV (EUR){nav_source_label}', col_today: f'{reported_nav:.5f}', col_prev: '', 'Change': ''},
            {'': 'Difference (EUR)', col_today: f'{nav_diff_eur:+.5f}', col_prev: '', 'Change': f'{nav_diff_pct:+.3f}%'},
        ]
    else:
        table_rows.append({'': 'Reported NAV', col_today: 'n/a', col_prev: '', 'Change': ''})

    table_df = pd.DataFrame(table_rows)

    ct = col_today
    def style_table(row, col_t=ct):
        styles = [''] * len(row)
        for i, col_name in enumerate(row.index):
            if col_name == col_t:
                styles[i] = 'font-weight: bold'
            elif col_name == 'Change':
                styles[i] = 'font-style: italic'
        return styles

    display(table_df.style
        .set_caption(f'{code} NAV calculation')
        .set_table_styles([{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '1.1em'), ('text-align', 'left'), ('padding-bottom', '8px')]}])
        .apply(style_table, axis=1)
        .hide(axis='index'))

    # Store for later cells
    fd['computed_nav'] = computed_nav
    fd['nav_diff_eur'] = nav_diff_eur
    fd['nav_diff_pct'] = nav_diff_pct
    fd['total_accrual'] = total_accrual
    fd['total_units'] = position_units

,04.03.2026,03.03.2026,Change
Total net assets as reported (EUR),"482,534,086.88","484,133,986.57",-0.33%
Repricing adjustment (EUR),"-1,812,670.77",,
Total net assets repriced (EUR),"480,721,416.11",,
Accrued mgmt fee est. (0.179% p.a.),"-9,430.04",,
Accrued depository fee est. (0.025% p.a.),"-1,317.04",,
Adjustment (outstanding invoices),"-9,676.30",,
Net assets after fees (EUR),"480,700,992.72","484,125,869.04",-0.71%
Units outstanding,"379,735,497.30","379,580,223.65",+0.04%
Computed NAV (EUR),1.26588,1.27542,-0.75%
Reported NAV (EUR) (from index data),1.27100,,


,04.03.2026,03.03.2026,Change
Total net assets as reported (EUR),"6,863,540.62","6,729,815.37",+1.99%
Repricing adjustment (EUR),"+1,807.12",,
Total net assets repriced (EUR),"6,865,347.74",,
Accrued mgmt fee est. (0.150% p.a.),-112.86,,
Accrued depository fee est. (0.025% p.a.),-18.81,,
Net assets after fees (EUR),"6,865,216.07","6,729,718.57",+2.01%
Units outstanding,"6,871,254.95","6,800,722.76",+1.04%
Computed NAV (EUR),0.99912,0.98956,+0.97%
Reported NAV (EUR) (from index data),0.99890,,
Difference (EUR),+0.00022,,+0.022%


## Day-over-Day Comparison

Comparing fund NAV change to MSCI ACWI previous-day change. No lag adjustment needed — both funds use NAV-date prices.

In [6]:
import yfinance as yf
import json
import ssl
from urllib.request import urlopen, Request

idx_data = raw_index.copy()
idx_data['Date'] = pd.to_datetime(idx_data['Date'])

# SSL context for MSCI API
ssl_ctx = ssl.create_default_context()
ssl_ctx.check_hostname = False
ssl_ctx.verify_mode = ssl.CERT_NONE

# Fetch MSCI ACWI Screened (Net EUR) from MSCI API
nav_dt = pd.to_datetime(NAV_DATE)
msci_start = (nav_dt - pd.Timedelta(days=10)).strftime('%Y%m%d')
msci_end = (nav_dt + pd.Timedelta(days=1)).strftime('%Y%m%d')
msci_url = f'https://app2.msci.com/products/service/index/indexmaster/getLevelDataForGraph?currency_symbol=EUR&index_variant=NETR&start_date={msci_start}&end_date={msci_end}&data_frequency=DAILY&index_codes=722376'
msci_raw = json.loads(urlopen(msci_url, context=ssl_ctx).read())
msci_levels = msci_raw['indexes']['INDEX_LEVELS']
msci_df = pd.DataFrame(msci_levels)
msci_df['Date'] = pd.to_datetime(msci_df['calc_date'].astype(str), format='%Y%m%d')
msci_df = msci_df.sort_values('Date')
print(f'MSCI ACWI Screened (722376) fetched: {len(msci_df)} rows')

# Fetch iShares MSCI ACWI Screened ETF (IG3A.DE) from Yahoo Finance
ig3a_start = (nav_dt - pd.Timedelta(days=10)).strftime('%Y-%m-%d')
ig3a_end = (nav_dt + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
ig3a_hist = yf.Ticker('IG3A.DE').history(start=ig3a_start, end=ig3a_end)
ig3a_hist.index = ig3a_hist.index.tz_localize(None)
print(f'IG3A.DE prices fetched: {len(ig3a_hist)} rows')

nav_dt_ts = pd.to_datetime(NAV_DATE)

def get_msci_change(prev_dt, nav_dt):
    on_nav = msci_df[msci_df['Date'] <= nav_dt]
    on_prev = msci_df[msci_df['Date'] <= prev_dt]
    if len(on_nav) > 0 and len(on_prev) > 0:
        t = on_nav.iloc[-1]
        p = on_prev.iloc[-1]
        if t['Date'] != p['Date']:
            return (t['level_eod'] - p['level_eod']) / p['level_eod'] * 100, p['Date'], t['Date']
    return None, None, None

def get_ig3a_change(prev_dt, nav_dt):
    ig3a_on_nav = ig3a_hist[ig3a_hist.index <= nav_dt]
    ig3a_on_prev = ig3a_hist[ig3a_hist.index <= prev_dt]
    if len(ig3a_on_nav) > 0 and len(ig3a_on_prev) > 0:
        t_val, p_val = ig3a_on_nav.iloc[-1]['Close'], ig3a_on_prev.iloc[-1]['Close']
        t_dt, p_dt = ig3a_on_nav.index[-1], ig3a_on_prev.index[-1]
        if t_dt != p_dt:
            return (t_val - p_val) / p_val * 100, p_dt, t_dt
    return None, None, None

for fund in FUNDS:
    code = fund['code']
    fd = fund_data[code]
    prev_date = fd['prev_date']
    computed_nav = fd.get('computed_nav')
    prev_nav = fd.get('prev_reported_nav')

    print(f'═══ {code} ═══')

    if computed_nav is None:
        print(f'Cannot compare: no computed NAV available\n')
        fd['fund_nav_change_pct'] = None
        continue

    if prev_nav is None or prev_date is None:
        print(f'Cannot compare: no previous reported NAV available\n')
        fd['fund_nav_change_pct'] = None
        continue

    fund_nav_change_pct = (computed_nav - prev_nav) / prev_nav * 100
    prev_dt_ts = pd.to_datetime(prev_date)

    print(f'Reported NAV {prev_date}: {prev_nav:.6f}  →  Computed NAV {NAV_DATE}: {computed_nav:.6f}  ({fund_nav_change_pct:+.2f}%)')

    rows = [
        {'Metric': 'NAV change (computed vs prev reported)', 'Value': f'{fund_nav_change_pct:+.2f}%'},
    ]

    msci_pct, msci_d0, msci_d1 = get_msci_change(prev_dt_ts, nav_dt_ts)
    ig3a_pct, ig3a_d0, ig3a_d1 = get_ig3a_change(prev_dt_ts, nav_dt_ts)

    if code == 'TUV100':
        # Custom blended benchmark: 60% MSCI ACWI Screened + 40% IG3A
        if msci_pct is not None:
            rows.append({'Metric': f'MSCI ACWI Screened NR EUR ({msci_d0.strftime("%m-%d")}→{msci_d1.strftime("%m-%d")})', 'Value': f'{msci_pct:+.2f}%'})
        if ig3a_pct is not None:
            rows.append({'Metric': f'IG3A.DE iShares ACWI ({ig3a_d0.strftime("%m-%d")}→{ig3a_d1.strftime("%m-%d")})', 'Value': f'{ig3a_pct:+.2f}%'})
        if msci_pct is not None and ig3a_pct is not None:
            blended = 0.6 * msci_pct + 0.4 * ig3a_pct
            rows.append({'Metric': 'Blended benchmark (60% MSCI + 40% IG3A)', 'Value': f'{blended:+.2f}%'})
            rows.append({'Metric': 'Tracking diff vs blended', 'Value': f'{fund_nav_change_pct - blended:+.2f}%'})

    elif code == 'TKF100':
        if ig3a_pct is not None:
            rows.append({'Metric': f'IG3A.DE iShares ACWI ({ig3a_d0.strftime("%m-%d")}→{ig3a_d1.strftime("%m-%d")})', 'Value': f'{ig3a_pct:+.2f}%'})
            rows.append({'Metric': 'Tracking diff vs IG3A.DE', 'Value': f'{fund_nav_change_pct - ig3a_pct:+.2f}%'})

    comparison = pd.DataFrame(rows)
    display(comparison.style.hide(axis='index'))

    fd['fund_nav_change_pct'] = fund_nav_change_pct
    fd['msci_change_pct'] = msci_pct
    fd['ig3a_change_pct'] = ig3a_pct
    print()

MSCI ACWI Screened (722376) fetched: 8 rows


IG3A.DE prices fetched: 8 rows
═══ TUV100 ═══
Reported NAV 2026-03-03: 1.266000  →  Computed NAV 2026-03-04: 1.265884  (-0.01%)


Metric,Value
NAV change (computed vs prev reported),-0.01%
MSCI ACWI Screened NR EUR (03-03→03-04),-0.38%
IG3A.DE iShares ACWI (03-03→03-04),+1.17%
Blended benchmark (60% MSCI + 40% IG3A),+0.24%
Tracking diff vs blended,-0.25%



═══ TKF100 ═══
Reported NAV 2026-03-03: 0.988300  →  Computed NAV 2026-03-04: 0.999121  (+1.09%)


Metric,Value
NAV change (computed vs prev reported),+1.09%
IG3A.DE iShares ACWI (03-03→03-04),+1.17%
Tracking diff vs IG3A.DE,-0.07%
